In [2]:
import numpy as np 
import torch 
import torch.nn as nn 
import math 

* 3 token example - "The" "cat" "sat" 
* Attention(Q, K, V) = softmax(Q.Transpose of K/ square root of dk)V
* Lets assume models embedding dimension is 4

# Scaled dot product attention

In [3]:
np.random.seed(2)
seq_len = 3 #Number of tokens 
d_k = 4 #Embedding dimension 

Q = np.random.randn(seq_len, d_k) # Generate a (3,4) matrix. So each row will represent the embeddings for each token. 3 tokens ~ 3 rows and each token 4 columns ~ embedding shape of 4
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

In [4]:
causal_mask = True
#calculate raw attention scores ~ Q.K transpose 
scores = np.dot(Q, K.T)

#Scale the scores to stabilise gradients ~ prevent covariate shift while applying softmax 
scaled_scores = scores/np.sqrt(d_k)
#shape for scores and scaled scores will be the same - like in this case (3,3)

#if causal masking is used 
if causal_mask:
    seq_len = scaled_scores.shape[0] # value here ll be 3
    # np.triu returns the upper triangle of an array 
    # k=1 excludes np.triu from working on the main diagonal so the token can attend to itself
    # the other ones we ll mask it so the attention scores wont be computable for them  
    mask = np.triu(np.ones((seq_len, seq_len)), k=1)
    scaled_scores = np.where(mask == 1, -np.inf, scaled_scores) # So the score matrix, upper right side triangle values except the diagonal will now be -inf cause we want to make the learnable matrixes -> Wq, Wk and Wv learn during backpropogation

# Now softmax is applied row wise 
# Row wise cause softmax returns the probability values over an array -> give 10 elements it will return the distribution for all 10 which ll result to 1 
# Since we need for each token their own probability scores we do it row wise
# Also this architecture makes it possible to compute the scores for multiple batch of tokens parallely 
# Although normalization has been done earlier we will also subtract the max value from each row for even better numerical stability (preventing overflow)
shifted_scores = scaled_scores - np.max(scaled_scores, axis=-1, keepdims=True) # Shape ll be (seq_len, seq_len) -> 3,3 | np.max applied on teh scaled_Scores and -1 means operation on the last shape value i.e column wise ~ one sum per row 
# Softmax is defined as 
    # Take the exponential of every element
    # Divide it by the sum of those exponentials
exp_scores = np.exp(shifted_scores) # here e to the power x is applied on every element individually where x is the element itself so the matrix shape does not change
attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

#As per the Attention(Q,K,V) formula now we have to multiply the attention_weights with V 
output = np.dot(attention_weights, V) #output shape be now (3,4) ~ (seq_len, d_k)

In [5]:
mask

array([[0., 1., 1.],
       [0., 0., 1.],
       [0., 0., 0.]])

In [6]:
scores, scaled_scores

(array([[-2.08380738,  1.55426205, -1.7951898 ],
        [ 1.87998556, -1.34207431,  3.06684728],
        [-0.0970001 , -1.43957461, -1.0538044 ]]),
 array([[-1.04190369,        -inf,        -inf],
        [ 0.93999278, -0.67103715,        -inf],
        [-0.04850005, -0.71978731, -0.5269022 ]]))

In [7]:
np.max(scaled_scores, axis=-1, keepdims=True)

array([[-1.04190369],
       [ 0.93999278],
       [-0.04850005]])

In [8]:
shifted_scores,np.exp(shifted_scores)

(array([[ 0.        ,        -inf,        -inf],
        [ 0.        , -1.61102993,        -inf],
        [ 0.        , -0.67128726, -0.47840215]]),
 array([[1.        , 0.        , 0.        ],
        [1.        , 0.19968185, 0.        ],
        [1.        , 0.5110503 , 0.61977291]]))

In [9]:
attention_weights

array([[1.        , 0.        , 0.        ],
       [0.83355433, 0.16644567, 0.        ],
       [0.46930219, 0.23983703, 0.29086078]])

* **Scores** can be defined as the raw projection of Q over transpose of K. Like the initial curiosity of the token "cat" shouting out its question Q. 
* **Attention Scores** is the normalized projection value. Scaling it down so it doesn't break down the gradients during training. Or in hindsight a step earlier does not cause the problem of covariate shift during applying softmax 
* **Attention Weights** Till now the projection values were still *numbers*. Applying softmax and squishing them into a probability distribution. Now you know how much "attention" each token is paying to each token
* **Output** Contextualized embeddings. Embeddings for each token containing contextualized information w.r.t and its position with other tokens

In [10]:
def scaled_dot_product_attention(Q, K, V, causal_mask):
    scores = np.dot(Q, K.T)
    scaled_scores = scores/np.sqrt(d_k)
    if causal_mask:
        seq_len = scaled_scores.shape[0]   
        mask = np.triu(np.ones((seq_len, seq_len)), k=1)
        scaled_scores = np.where(mask == 1, -np.inf, scaled_scores) 
    shifted_scores = scaled_scores - np.max(scaled_scores, axis=-1, keepdims=True) 
    exp_scores = np.exp(shifted_scores) 
    attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    output = np.dot(attention_weights, V) 
    
    return output, attention_weights

In [11]:
#Running bidirectional attention 
out, weights = scaled_dot_product_attention(Q, K, V, False)
print("Bidirectional attention weights", weights)

# Causal masked attention
out, weights_masked = scaled_dot_product_attention(Q,K,V, True)
print("\n\nCausal masked attention weights are ", weights_masked) #upper right hand or triangle of these weights ll be zero meaning that the future tokens will be blocked from leaking backwards. The current token cannot "cheat" or look at the future tokens 

Bidirectional attention weights [[0.12017578 0.74099227 0.13883196]
 [0.33224025 0.06634235 0.6014174 ]
 [0.46930219 0.23983703 0.29086078]]


Causal masked attention weights are  [[1.         0.         0.        ]
 [0.83355433 0.16644567 0.        ]
 [0.46930219 0.23983703 0.29086078]]


# Learnable parameters Wq, Wk and Wv

* These are the matrices which ll learn during training so they have to be defined as tensors with gradient updation possible 
* Each **sequence** is a collection of tokens together. Like in a sentence or para ig
* **Batch size** is the number of sequences processed together 
* **Sequence Length** is the number of tokens in a sequence 
* **d_model** is the embedding size of each token 
* **d_k** is the dimension of the query, key and value vectors 
    * As per my understanding for a single head attention d_k == d_model but not in MHA. 
    * d_model = 768 and there are 12 heads. 768/12 = 24. Each head gets d_k=24

In [12]:
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__() 
        self.d_k = d_k 

        # Conceptually I thought we could just do nn.Parameter(torch.randn(d_model, d_k) 
        # But we need to use Xavier/Glorot initialization for training stability
        # Else the variance will keep on increasing near exponentially 
        self.W_Q = nn.Parameter(torch.randn(d_model, d_k) * math.sqrt(2.0/(d_model+d_k)) )
        self.W_K = nn.Parameter(torch.randn(d_model, d_k) * math.sqrt(2.0/(d_model+d_k)) )
        self.W_V = nn.Parameter(torch.randn(d_model, d_k) * math.sqrt(2.0/(d_model+d_k)) )

    def forward(self, X, causal_mask=False):
        batch_size, seq_len, _ = X.shape 
        # Each X is [batch size, seq lenght, d_k]

        # project the inputs to Q,K,V spaces using matrix multiplications
        # matmul handles the broadcasting for batch shapes 
        Q = torch.matmul(X, self.W_Q)
        K = torch.matmul(X, self.W_K) 
        V = torch.matmul(X, self.W_V) 

        #Calculate raw scores 
        scores = torch.matmul(Q, K.transpose(-2, -1)) #Since now they are 3D tensors we need to choose which dimensions to "swap" transpose so if the shape indexes are 0,1,2 we are swapping 2 and 1
        #Scale the scores 
        scaled_scores = scores / math.sqrt(self.d_k) 

        if causal_mask:
            mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(X.device)
            scaled_scores = scaled_scores.masked_fill(mask, float("-inf"))

        attention_weights = torch.softmax(scaled_scores, dim=-1) # Shape: [Batch_Size, Seq_Len, Seq_Len]
        output = torch.matmul(attention_weights, V)

        return output, attention_weights


In [13]:
torch.manual_seed(32)

B, S, d_model, d_k = 2, 3, 4, 4 
mock_input = torch.randn(B, S, d_model)

attention_layer = SingleHeadAttention(d_model, d_k)
output, weights = attention_layer(mock_input, causal_mask=True)

print("Output tensor shape is ", output.shape)
print("\nAttn weights shape is ", weights.shape)
print("\nBatch 1 sample attention weight matrix is ", weights[0])


Output tensor shape is  torch.Size([2, 3, 4])

Attn weights shape is  torch.Size([2, 3, 3])

Batch 1 sample attention weight matrix is  tensor([[1.0000, 0.0000, 0.0000],
        [0.3698, 0.6302, 0.0000],
        [0.2089, 0.3375, 0.4536]], grad_fn=<SelectBackward0>)


* Similar to teh numpy implementation of scaled dot product attention, the attn weights have 0 in the upper right triangle of the matrix excluding the diagonal

In [14]:
attention_layer = SingleHeadAttention(d_model, d_k)
output, weights = attention_layer(mock_input, causal_mask=False)

print("Output tensor shape is ", output.shape)
print("\nAttn weights shape is ", weights.shape)
print("\nBatch 1 sample attention weight matrix is ", weights[0])

Output tensor shape is  torch.Size([2, 3, 4])

Attn weights shape is  torch.Size([2, 3, 3])

Batch 1 sample attention weight matrix is  tensor([[0.5436, 0.1493, 0.3071],
        [0.3231, 0.3671, 0.3098],
        [0.3578, 0.3718, 0.2704]], grad_fn=<SelectBackward0>)


* On removing causal masking they are not 0 since its biderectional contextual understanding work so no need of hiding the future tokens

# Multi Head Attention 

* Instead of running on attention on one head the sequence is split across multiple 'heads'
* Each head runs its own attention initialized with their own random numbers
* This should make each head learn different patterns, pos and relationships in the sequence. 
* From an engineering pov it also makes it more efficient in running different heads in parallel - using accelerator cores parallelism
* Taking the values from earlier 
    * d_model =4, num_heads =2, therefore dimension per head would be 4/2 = 2 

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__():
        self.d_model = d_model 
        self.num_heads = num_heads 

    # We will check if the dimensions are viable
    # If embedding dimension per head is possible 
    assert d_model % num_heads == 0, "d_model must be completely divisible by number of heads"
    self.d_k = d_model // num_heads 

    